In [34]:
import os
from openai import OpenAI
import pandas as pd
import numpy as np
import re
from pypinyin import lazy_pinyin
from rapidfuzz import fuzz
import math
from uuid import uuid4 as uuid
from dotenv import load_dotenv
import subprocess
from tqdm import tqdm
import json
load_dotenv(".env")

root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
df = pd.read_csv("songs.csv")
df.tail(1)

,code,type,title,lyrics,pinyin
490,UNK-4,Hymn,以马内利来临,以马内利恳求降临，\n救赎解放以色列民；\n沦落异邦寂寞伤心，\n引颈渴望神子降临。\n欢欣...,yi ma nei li ken qiu xiang lin jiu shu jie fan...


In [2]:
def pinyin(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    result = " ".join(lazy_pinyin(text))
    return re.sub(r"\s+", " ", result).strip()


def windows(tokens, size, step):
    if len(tokens) <= size:
        yield " ".join(tokens)
    else:
        for i in range(0, len(tokens) - size + 1, step):
            yield " ".join(tokens[i:i+size])


def best_window_score(query_py, lyrics_py, size=50, step=10):
    query_tokens = query_py.split()
    lyric_tokens = lyrics_py.split()
    score = max(
        fuzz.ratio(qw, lw)
        for qw in windows(query_tokens, size, step)
        for lw in windows(lyric_tokens, size, step)
    )
    return score / 100


def get_duration(filepath):
    duration = float(subprocess.check_output([
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        filepath,
    ]).decode().strip())
    return duration


def split_to_limit(filepath, limit=26_214_400, margin=0.90, out_dir="tmp"):
    size = os.path.getsize(filepath)
    if size <= limit:
        return [filepath]
    os.makedirs(out_dir, exist_ok=True)
    duration = get_duration(filepath)
    bitrate_kbps = 128
    chunk_seconds = max(1, int(limit * margin * 8 / (bitrate_kbps * 1000)))
    chunk_paths = []
    
    for start in range(0, math.ceil(duration), chunk_seconds):
        chunk_path = os.path.join(out_dir, f"{uuid()}.mp3")
        subprocess.run([
            "ffmpeg",
            "-y",
            "-ss", str(start),
            "-t", str(chunk_seconds),
            "-i", filepath,
            "-vn",
            "-c:a", "libmp3lame",
            "-b:a", f"{bitrate_kbps}k",
            chunk_path,
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        chunk_paths.append(chunk_path)
        print(
            f"chunk {len(chunk_paths)}: "
            f"{os.path.getsize(chunk_path):,} bytes "
            f"(limit {limit:,}) -> {chunk_path}"
        )
    return chunk_paths

In [3]:
def match_zoom_to_sq(d, tol=10, verbose=False):
    files = sorted(os.listdir(f"{root}/{d}"))
    zoom_files = [f for f in files if f.startswith("ZOOM")]
    sq_files = [f for f in files if not f.startswith("ZOOM")]
    
    if not zoom_files or not sq_files:
        return {}

    durations = {f: int(get_duration(f"{root}/{d}/{f}")) for f in files}
    if verbose:
        print(json.dumps(durations, indent=2, ensure_ascii=False))

    reduced_sq_files = []
    for f in sq_files:
        other_files = [x for x in reduced_sq_files if not x.startswith("SQ")]
        if f.startswith("SQ") or not any(durations[f] == durations[x] for x in other_files):
            reduced_sq_files.append(f)
    sq_files = reduced_sq_files

    if verbose:
        print(zoom_files)
        print(sq_files)

    # If same number of files, just pair in order
    if len(zoom_files) == len(sq_files):
        ordered_matches = dict(zip(zoom_files, sq_files))
        # Make sure the sizes match though
        if all(abs(durations[z] - durations[s]) < tol for z, s in ordered_matches.items()):
            return ordered_matches

    # Otherwise, take the largest filesize (P&W) pair as reference
    matches = {}

    def traverse(ref_zoom_idx, ref_sq_idx, zoom_list, sq_list):
        z_idx = ref_zoom_idx + 1
        s_idx = ref_sq_idx + 1
        while z_idx < len(zoom_list) and s_idx < len(sq_list):
            zoom_file = zoom_list[z_idx]
            for i, sq_file in enumerate(sq_list):
                if sq_file.startswith("SQ") and i < s_idx:
                    continue
                if verbose:
                    print(f"{i=} {s_idx=} {zoom_file} ({durations[zoom_file]}) : {sq_file} ({durations[sq_file]})")
                if abs(durations[zoom_file] - durations[sq_file]) > tol:
                    continue
                matches[zoom_file] = sq_file
                if sq_file.startswith("SQ"):
                    s_idx = i + 1
                break
            z_idx += 1

    def prune_same_values(d):
        counts = {v: len([k for k in d if d[k] == v]) for v in d.values()}
        return {k: v for k, v in d.items() if counts[v] == 1}

    largest_zoom_file = max(zoom_files, key=lambda f: durations[f])
    largest_sq_file = max(sq_files, key=lambda f: durations[f])
    if not largest_sq_file.startswith("SQ"):
        # since order can't be gleaned from filename, just traverse the whole list
        traverse(-1, -1, zoom_files, sq_files)
        return prune_same_values(matches)

    if abs(durations[largest_zoom_file] - durations[largest_sq_file]) <= tol:
        matches[largest_zoom_file] = largest_sq_file
        zoom_idx = zoom_files.index(largest_zoom_file)
        sq_idx = sq_files.index(largest_sq_file)
    else:
        cost = np.array([
            [abs(durations[z] - durations[s]) for s in sq_files]
            for z in zoom_files
        ])
        zoom_idx, sq_idx = np.unravel_index(np.argmin(cost), cost.shape)
        matches[zoom_files[zoom_idx]] = sq_files[sq_idx]

    traverse(zoom_idx, sq_idx, zoom_files, sq_files)
    traverse(len(zoom_files) - zoom_idx - 1, len(sq_files) - sq_idx - 1, zoom_files[::-1], sq_files[::-1])

    return prune_same_values(matches)

# matches = match_zoom_to_sq("2026-04-11")
# print(json.dumps(matches, indent=2, ensure_ascii=False))

In [52]:
num = max([int(code.split("-")[1]) for code in df.loc[df.code.str.startswith("UNK")].code])
lyrics = """你仰脸保守你所爱的 
你护庇永远不离不弃 
你将我从淤泥里捧起 
放我在你的手心

我凭着信心领取 
你丰盛的应许 
世界也不能夺去 
神美好的旨意

我凭着信心领取 
你恩典永不止息 
看见美好应许成就 
荣耀全都归于你

我等候主我相信 
你爱我永不放弃 
世界也不能夺去 
神美好的旨意

我等候主我相信 
你旨意高过我的 
看见美好应许成就 
我全心全意敬拜你"""
df.loc[len(df)] = {
    "code": f"UNK-{num + 1}",
    "type": "PnW",
    "title": "丰盛的应许",
    "lyrics": lyrics,
    "pinyin": pinyin(lyrics),
}
df.to_csv("songs.csv", index=False)
df.tail(1)

,code,type,title,lyrics,pinyin
491,UNK-5,PnW,丰盛的应许,你仰脸保守你所爱的 \n你护庇永远不离不弃 \n你将我从淤泥里捧起 \n放我在你的手心\n\...,ni yang lian bao shou ni suo ai de ni hu bi yo...


In [48]:
for d in sorted(os.listdir(root), reverse=True):
    if not d.startswith("2025-12"):
        continue
    files = sorted(os.listdir(f"{root}/{d}"))
    for f in files:
        if bool(re.search(r'[\u4e00-\u9fff]', f)):
            continue
        filepath = f"{root}/{d}/{f}"
        duration = get_duration(filepath)
        mins, secs = int(duration // 60), int(duration % 60)
        print(f"{filepath} {mins:02d}:{secs:02d}")

/mnt/NextcloudSacmData/sacm.av/files/Recordings/2025-12-27/(SAT) SQ-ST163.mp3 03:39
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2025-12-27/(SAT) ZOOM0156.mp3 03:39
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2025-12-13/(SAT) ZOOM0139.mp3 00:04


In [ ]:
print(split_to_limit(f"{root}/2025-12-28/(SAT) SQ-ST056.mp3"))

['/mnt/NextcloudSacmData/sacm.av/files/Recordings/2025-10-18/(SAT) SQ-ST056.mp3']


In [56]:
lyrics = ""
for filepath in tqdm([f"{root}/2025-12-28/SQ-ST165_将天敞开.mp3"]):
    audio_file = open(filepath, "rb")
    transcription = client.audio.transcriptions.create(
        model="gpt-4o-transcribe", 
        # model="whisper-1", 
        file=audio_file,
        language="zh",
    )
    lyrics += transcription.text
lyrics

100%|██████████| 1/1 [00:19<00:00, 19.97s/it]


'无论你的日子今年过得怎么样我们今天来到来到主的面前我们都献上我们的感受我们的赞美我们用最热烈的歌声去赞美我们天上福因为祂是独一的主无论顺景或者异景祂都会带领我们进入新的一年我们把这个祈求这个赞美我们都献给我们天上的父我们一起来同唱张天敞开我们一起来拍手好吗张天敞开张天敞开你的荣耀张下来张天敞开你的同在张下来张天敞开你的荣耀张下来万国赞叹你你是荣耀君王天上地下和一经百欢呼耶稣基督圣洁高让荣耀归于你天上地下在永恒里敬拜哈雷路亚哈雷路亚是的我们把我们赞美我们的欢呼都传到天上去送给我们至上父至上的主再次张天敞开张天敞开你的荣耀张下来张天敞开你的同在张下来张天敞开你的荣耀张下来万国赞叹你你是荣耀君王天上地下天上地下和一经百欢呼耶稣基督圣洁高让荣耀归于你天上地下在永恒里敬拜哈雷路亚哈雷路亚神就在这里神就在这里我们欢迎你让一切招点转向你神就在这里神就在这里我们欢迎你抱走前进百步停息张天敞开你的荣耀张下来张天敞开你的同在张下来张天敞开你的荣耀张下来万国赞叹你你是荣耀君王天上地下和一经百欢呼耶稣基督圣洁高让荣耀归于你天上地下在永恒里敬拜哈雷路亚哈雷路亚天上地下和一经百欢呼耶稣基督圣洁高让荣耀归于你天上地下在永恒里敬拜哈雷路亚我们把我们的掌声我们的欢呼送给我们至上父至上子因为祂带领我们渡过每一个日子平安或不平安祂都与我们同在哈雷路亚是的无论平安顺或逆祂都与我们同在祂还赐给我们耶稣基督耶稣基督这一份丰盛的应许来到我们每个人心中你让念保守你所爱的你无比永远不离不弃你将我送于你你捧起放我在你的手心我凭着信心领取你丰盛的应许世界也不能夺取神美好的旨意我凭着信心领取你恩典永不窒息看见美好应许成就荣耀全都归于你我在唱一次的时候是想歌词是想我们新的一年如何回应耶稣你让念保守你所爱的你无比永远不离不弃你将我送于你你捧起放我在你的手心我们凭着信心领取我凭着信心领取你丰盛的应许世界也不能夺取神美好的旨意我凭着信心领取你恩典永不窒息看见美好应许成就荣耀全都归于你我能hold住我相信你爱我永不放弃世界也不能夺取神美好的旨意我能hold住我相信你执意高过火地看见美好应许成就我愿意愿意敬拜你看见美好看见美好应许成就我愿意愿意敬拜你各位弟兄姐妹我们一起闭上我们的眼睛我们做一个简单的祷告亲爱主我们感谢你感谢你赐给我们耶稣基督作为我们人生中最丰盛的应许他来到这个世界洗尽我们的罪让我们能够重新的与你连接有永生的盼望我们

In [57]:
titles = {}
title_to_last_chunk_idx = {}

query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

chunk_size = 120
for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
    chunk = query_lyrics[start:start + chunk_size]
    if len(chunk) < 50:
        continue
    query_py = pinyin(chunk)
    scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
    best_idx = np.argmax(scores)
    best_title = df.iloc[best_idx]["title"]
    best_score = scores[best_idx]
    print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
    if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
        titles[best_title] = max(titles[best_title], best_score) * 1.2
    else:
        titles[best_title] = best_score
    title_to_last_chunk_idx[best_title] = i

print(f"{titles=}")
final_titles = [title for title, score in titles.items() if score > 0.7]
print(f"Songs: {'_'.join(final_titles)}")

[0:120] best_title='宣扬祂的荣耀', best_score=0.5765306122448979
[120:240] best_title='将天敞开', best_score=0.88
[240:360] best_title='将天敞开', best_score=0.8802902055622732
[360:480] best_title='将天敞开', best_score=0.8512696493349456
[480:600] best_title='爱使我们勇敢+我们爱', best_score=0.5659411011523687
[600:720] best_title='丰盛的应许', best_score=0.9214929214929215
[720:840] best_title='丰盛的应许', best_score=0.9069767441860467
[840:960] best_title='丰盛的应许', best_score=0.6257982120051085
[960:1080] best_title='爱使我们勇敢+我们爱', best_score=0.5791505791505791
titles={'宣扬祂的荣耀': 0.5765306122448979, '将天敞开': 1.2676178960096733, '爱使我们勇敢+我们爱': 0.5791505791505791, '丰盛的应许': 1.326949806949807}
Songs: 将天敞开_丰盛的应许


In [55]:
print(df.loc[df.title == "古人喜乐"].iloc[0].lyrics)

古代人们心喜乐，仰望明星作引导；
欣然欢迎它光彩，灿烂辉煌前路照；
求主使我亦如此，追随景星行主道。

他们欢然行远路，走向卑微马槽旁；
屈膝虔诚恭敬拜，天人崇敬一大君王；
求主使我亦虔诚，永寻耶稣施恩座。

他们敬将好礼物，奉献婴孩在座前；
求使我亦欣然，清洁无罪灵魂安；
求主使我亦献重礼，向主奉献身心灵。

恳求救主领我众，奔走窄路赴窄门；
来待到世成过去，有主赎救灵魂近主；
求主携我不需星光照，有主荣耀永光明。


In [ ]:
chunk = query_lyrics[2640:2760]
print("Query:", chunk)
query_pinyin = pinyin(chunk)
for t in ["宁静谷"]:
    inds = df.loc[df.title == t].index
    for idx in inds:
        print(f"[{idx}] {t}: {df.pinyin[idx]}")
        fuzz_score = best_window_score(query_pinyin, df.pinyin[idx], size=100, step=3)
        print(f"{fuzz_score}")

Query:  我学会了信靠他 依靠他 有一次当我 向一位朋友 倾诉我的挣扎时 他推荐我 他推荐给我一首 藏民之群的歌 叫《宁静谷》 歌词中写道 生活中的仓促 生命里的难处 只愿向他来倾诉 平安祝福在这谷 我觉得这首歌 正好讲述了 那段时期 上帝如何 把
[77] 宁静谷: zai wo xin ling shen chu you yi zuo ning jing gu wo he wo qin ai de zhu zai qi zhong an ran man bu sheng huo zhong de cang cu sheng ming li de nan chu zhi yuan xiang ta lai qing su ping an zhu fu zai zhe gu wo yu wo zhu xiang yue zhi chu chang yang zhe fen ning jing an xiang jiu xiang shi zai tian tang wo yu wo zhu xiang yue zhi chu zhu ling wo guo si yin you gu shi wo xi le zou ren sheng lu
score=0.5467158003484595, fuzz_score=0.5852417302798982, 0.2868419756502333


In [ ]:
def get_titles(filepath):
    print(f"Processing {filepath}")

    cropped_paths = split_to_limit(filepath)
    lyrics = ""
    for filepath in tqdm(cropped_paths):
        audio_file = open(filepath, "rb")
        transcription = client.audio.transcriptions.create(
            # model="gpt-4o-transcribe", 
            model="whisper-1", 
            file=audio_file,
            language="zh",
        )
        lyrics += transcription.text

    if not lyrics:
        return []

    titles = {}
    title_to_last_chunk_idx = {}

    query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

    chunk_size = 120
    for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
        chunk = query_lyrics[start:start + chunk_size]
        if len(chunk) < 50:
            continue
        query_py = pinyin(chunk)
        scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
        best_idx = np.argmax(scores)
        best_title = df.iloc[best_idx]["title"]
        best_score = scores[best_idx]
        print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
        if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
            titles[best_title] = max(titles[best_title], best_score) * 1.2
        else:
            titles[best_title] = best_score
        title_to_last_chunk_idx[best_title] = i

    print(f"{titles=}")
    duration = get_duration(filepath)
    if duration > 3 * 60:
        final_titles = [title for title, score in titles.items() if score > 0.7]
    else:
        best_title = max(titles, key=titles.get)
        final_titles = [best_title] if titles[best_title] > 0.7 else []
    return final_titles

In [ ]:
final_titles = get_titles(f"{root}/2026-03-29/SQ-ST303.mp3")
print(f"Songs: {'_'.join(final_titles)}")

In [76]:
for d in ["2026-06-27", "2026-06-20", "2026-06-14", "2026-05-07", "2026-04-11", "2026-03-08", "2026-03-07", "2026-02-14", "2026-02-08", "2026-02-07", "2026-02-01", "2026-01-25"]:
    matches = match_zoom_to_sq(d, verbose=False)
    print(d, json.dumps(matches, indent=2, ensure_ascii=False))

2026-06-27 {
  "ZOOM0429_你的爱_一颗谦卑的心.mp3": "SQ-ST415_你的爱_一颗谦卑的心.mp3",
  "ZOOM0427_将天敞开.mp3": "SQ-ST413_将天敞开.mp3",
  "ZOOM0426_将你最好的献给主.mp3": "SQ-ST412_将你最好的献给主.mp3"
}
2026-06-20 {
  "ZOOM0413_宝贵十架_一生一世.mp3": "SQ-ST398_宝贵十架_一生一世.mp3",
  "ZOOM0414_一生一世.mp3": "SQ-ST399_一生一世.mp3",
  "ZOOM0415_一生一世.mp3": "SQ-ST400_一生一世.mp3",
  "ZOOM0412_耶稣基督是主.mp3": "SQ-ST396_耶稣基督是主.mp3",
  "ZOOM0411_基督精兵前进.mp3": "SQ-ST395_基督精兵前进.mp3",
  "ZOOM0410_伟大的救主.mp3": "SQ-ST393_伟大的救主.mp3"
}
2026-06-14 {
  "ZOOM0405_我們的神_何等恩典.mp3": "SQ-ST389_我們的神_何等恩典.mp3",
  "ZOOM0406_一颗谦卑的心.mp3": "SQ-ST390_一颗谦卑的心.mp3",
  "ZOOM0408_一颗谦卑的心.mp3": "SQ-ST391_一颗谦卑的心.mp3",
  "ZOOM0409_歌颂主爱.mp3": "SQ-ST392_歌颂主爱.mp3"
}
2026-05-07 {}
2026-04-11 {
  "ZOOM0306_你真伟大.mp3": "SQ-ST306_你真伟大.mp3",
  "ZOOM0307_救主耶稣万福恩源.mp3": "SQ-ST307_救主耶稣万福恩源.mp3",
  "ZOOM0312_点燃.mp3": "SQ-ST309_点燃.mp3"
}
2026-03-08 {
  "ZOOM0266_尊贵全能神_无价至宝_我的盼望在于祢.mp3": "SQ-ST268_尊贵全能神_无价至宝_我的盼望在于祢.mp3"
}
2026-03-07 {
  "ZOOM0259_耶稣领我.mp3": "SQ-ST263_耶稣领我.mp3",
  "ZOOM0260_尊贵全能神.mp3